In [1]:
import datetime

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el último día del mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_actual = ultimo_dia_mes_anterior.month
mes_anterior = ultimo_dia_mes_anterior.month -1
anio_anterior = ultimo_dia_mes_anterior.year

#Asignr variables
MES_ACTUAL = f"{anio_anterior}{mes_actual:02d}"
MES_SQL_SP_LIQ_ACTUAL = f"'{anio_anterior}{mes_anterior:02d}'"
PERIODO_ALMACEN = f"{anio_anterior}{mes_anterior:02d}"
PERIODO_NORMAL_float =f"{anio_anterior}{mes_anterior:02d}"
peso_calificacion_ojo =0.07 #PORCENTAJE AFECTACIONES CAMBIAR SI CAMBIA EL PORCENTAJE DE AFECTACION

#MOSTRAR RESULTADOS
print("Mes actual =",MES_ACTUAL)
print("SP =", MES_SQL_SP_LIQ_ACTUAL)
print("PERIODO FINANCIERO =",PERIODO_ALMACEN)
print("PERIODO_NOMAL ENTERO =",PERIODO_NORMAL_float)
print("peso afectacion=",peso_calificacion_ojo)

Mes actual = 202610
SP = '202609'
PERIODO FINANCIERO = 202609
PERIODO_NOMAL ENTERO = 202609
peso afectacion= 0.07


In [2]:
#Importacion de librerias
import pandas as pd
from pandas import json_normalize
import numpy as np
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
from unidecode import unidecode
from googleapiclient.discovery import build
from google.oauth2 import service_account

## --------PASO A PASO RELIQUIDACION ---------------

In [3]:
#SE TOMAN LOS OK PARA SABER LA CANTIDAD DE VENTAS A RELIQUIDAR
#-------------------------------------------------------------------------------
host2_8 = '172.16.1.33'
port2_8 = '1433'
database2_8= 'CUN_REPOSITORIO'

try:
    url2_8 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_8, port=port2_8, db=database2_8)
    engine2_8= create_engine(url2_8) 
    print("Conexion a la base de datos realizada")
    Etiquetas_mes_acutual = pd.read_sql(f'''
        select * from COMISIONES_CONTACT.ETIQUETASRELIQUIDACION
                             where Periodo_Pago ={MES_SQL_SP_LIQ_ACTUAL} and EstadoFinal = 'OK'
                            ''', engine2_8)
    Etiquetas_mes_acutual2 = Etiquetas_mes_acutual.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_8' in locals():
        engine2_8.dispose()
        print("Conexión cerrada")
Etiquetas_mes_acutual['Llave_etiq_per']=Etiquetas_mes_acutual['Doc_Estudiante'].astype(str)+Etiquetas_mes_acutual['PERIODO'].astype(str)
Etiquetas_mes_acutual

Conexion a la base de datos realizada
Conexión cerrada


,Doc_Estudiante,PERIODO,Documento_del_Asesor,Propietario_del_registro,Programa_vendido,Correo_Asesor,Periodo_Pago,EST_MATRICULADO,NCR_ANULACION,TIPO,ORDEN,VAL_LIQUIDADO,NUEVO,ESTADO_PAGO,Documento_Estudiante_zoho,Motivo,Propietario_de_Fase_Name,EstadoFinal,Llave_etiq_per
0,1007495024,26V05,1016001445,Contact Norvey Ibarra Marquez,202609_DIRECCION Y PRODUCCION DE MEDIOS AUDIOV...,NORVEY_IBARRA@CUN.EDU.CO,202609.0,1-Activo,0.0,Pregrado,3852464.0,NaN,NUEVO,PAGO,1007495024,ok,None,OK,100749502426V05
1,80136252,26V05,1064804396,Contact Cristian Camilo Sanchez Arias,202609_COMUNICACION SOCIAL,CRISTIAN_SANCHEZ@CUN.EDU.CO,202609.0,1-Activo,0.0,Pregrado,3896872.0,2971800.0,NUEVO,PAGO,80136252,ok,None,OK,8013625226V05


In [4]:
# Obtener cédulas únicas
cedulas_unicas = Etiquetas_mes_acutual['Doc_Estudiante'].unique()

In [5]:
# Dividir en 5 partes equilibradas
def dividir_en_partes(lista, partes):
    k, m = divmod(len(lista), partes)
    return [lista[i*k + min(i, m):(i+1)*k + min(i+1, m)] for i in range(partes)]

cedulas_partes = dividir_en_partes(cedulas_unicas, 5)

# Crear diccionario de partes formateadas para consulta SQL
cedulas_dict = {
    i + 1: ", ".join([f"''{cedula}''" for cedula in parte])
    for i, parte in enumerate(cedulas_partes)
}
cedulas_dict

{1: "''1007495024''", 2: "''80136252''", 3: '', 4: '', 5: ''}

In [6]:
import time
import pandas as pd
from sqlalchemy import create_engine

Recibos_Caja = pd.DataFrame()

for i in range(1, 6):

    # ---------------------------------------------------------
    # CÉDULAS DEL BLOQUE
    # ---------------------------------------------------------
    cedulas_consulta = cedulas_dict[i]

    print(f"\n🔎 Procesando bloque {i}")
    print(f"Cédulas: {cedulas_consulta}")

    host2_9 = '172.16.1.33'
    port2_9 = '1433'
    database2_9 = 'CUN_REPOSITORIO'

    engine2_9 = None

    try:

        # ---------------------------------------------------------
        # CONEXIÓN
        # ---------------------------------------------------------
        url2_9 = (
            f"mssql+pyodbc://@{host2_9}:{port2_9}/{database2_9}"
            "?trusted_connection=yes&driver=SQL+Server"
        )

        engine2_9 = create_engine(url2_9)

        print(f"✅ Conexión realizada - bloque {i}")

        # ---------------------------------------------------------
        # QUERY
        # ---------------------------------------------------------
        query = f"""
        SELECT *
        FROM OPENQUERY([172.16.1.175], '
            SELECT
                r.DOC_ALUM,
                r.NOMBRE_ESTUDIANTE,
                r.Periodo,
                r.Documento,
                r.orden,
                r.valor_orden,
                r.valor_ingreso_directo,
                r.fecha_de_pago,
                t.documento AS documento_relacionado,
                t.orden AS orden_relacionada

            FROM iceberg.r_recaudo_pagos r

            LEFT JOIN ICEBERG.cunt_tramite_externo t
                ON t.periodo = r.periodo
                AND t.identificacion = r.doc_alum
                AND t.orden_inicial = r.orden
                AND t.documento_inicial = r.documento

            WHERE r.doc_alum IN ({cedulas_consulta})
        ')
        """

        # ---------------------------------------------------------
        # EJECUTAR
        # ---------------------------------------------------------
        df = pd.read_sql_query(query, engine2_9)

        print(f"📊 Bloque {i}: {len(df)} registros encontrados")

        # ---------------------------------------------------------
        # CONCATENAR
        # ---------------------------------------------------------
        Recibos_Caja = pd.concat(
            [Recibos_Caja, df],
            ignore_index=True
        )

    except Exception as e:

        print(f"❌ Error en bloque {i}: {e}")

    finally:

        if engine2_9 is not None:
            engine2_9.dispose()
            print(f"🔌 Conexión cerrada - bloque {i}")

    time.sleep(2)

print("\n===================================")
print(f"✅ TOTAL REGISTROS: {len(Recibos_Caja)}")
print("===================================")


🔎 Procesando bloque 1
Cédulas: ''1007495024''
✅ Conexión realizada - bloque 1
📊 Bloque 1: 5 registros encontrados
🔌 Conexión cerrada - bloque 1

🔎 Procesando bloque 2
Cédulas: ''80136252''
✅ Conexión realizada - bloque 2
📊 Bloque 2: 1 registros encontrados
🔌 Conexión cerrada - bloque 2

🔎 Procesando bloque 3
Cédulas: 
✅ Conexión realizada - bloque 3
❌ Error en bloque 3: (pyodbc.ProgrammingError) ('42000', '[42000] [Microsoft][ODBC SQL Server Driver][SQL Server]Cannot process the object "\n            SELECT\n                r.DOC_ALUM,\n                r.NOMBRE_ESTUDIANTE,\n                r.Periodo,\n                r.Documento,\n                r.orden,\n                r.valor_orden,\n                r.valor_ingreso_directo,\n                r.fecha_de_pago,\n                t.documento AS documento_relacionado,\n                t.orden AS orden_relacionada\n\n            FROM iceberg.r_recaudo_pagos r\n\n            LEFT JOIN ICEBERG.cunt_tramite_externo t\n                ON t.pe

In [7]:
Recibos_Caja['DOC_ALUM'] = Recibos_Caja['DOC_ALUM'].fillna(0).astype(int).astype(str)
#aca filtro el dataframe solo con los pagos que corresponden a los mismo periodos que en etiquetas
Recibos_Caja['Llave_per_caja']=Recibos_Caja['DOC_ALUM'].astype(str)+Recibos_Caja['PERIODO'].astype(str)
Recibos_Caja = Recibos_Caja[Recibos_Caja['Llave_per_caja'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
print(Recibos_Caja.shape)
Recibos_Caja 

(3, 11)


,DOC_ALUM,NOMBRE_ESTUDIANTE,PERIODO,DOCUMENTO,ORDEN,VALOR_ORDEN,VALOR_INGRESO_DIRECTO,FECHA_DE_PAGO,DOCUMENTO_RELACIONADO,ORDEN_RELACIONADA,Llave_per_caja
1,1007495024,MONTAÑEZ DUEÑAS ANLLY YAJAIRA,26V05,RPVI,219397.0,64083.0,64083.0,28/09/2026,FAMA,3912769.0,100749502426V05
4,1007495024,MONTAÑEZ DUEÑAS ANLLY YAJAIRA,26V05,FAMA,3912769.0,4037200.0,126739.0,30/10/2026,None,NaN,100749502426V05
5,80136252,GONZÀLEZ CRISTANCHO FELIPE ANDRÈS,26V05,RPVI,213978.0,57533.0,57533.0,18/09/2026,FAMA,3896872.0,8013625226V05


In [8]:
Recibos_Caja = Recibos_Caja.drop('Llave_per_caja', axis=1)
#---------------------------------------------------------------------------
#ACA ORNANIZO LA TABLA DE RECIBOS DE CAJA
#---------------------------------------------------------------------------
Recibos_Caja_t=Recibos_Caja[['DOC_ALUM','VALOR_INGRESO_DIRECTO']]
df_resumen_recibos = Recibos_Caja_t.groupby('DOC_ALUM', as_index=False)['VALOR_INGRESO_DIRECTO'].sum()

Aqui no hay duplicados en el df de recibos caja por ende no habrá multiples coincidencias

In [9]:
#Unicidad de cedulas para consulta de valor a pagar por el estudiante en recibos contact
#------------------------------------------------------------------------------------------------------------
Cedulas_filtroz = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
cedulas_para_zoho = ", ".join([f"'{cedulaz}'" for cedulaz in Cedulas_filtroz])

In [10]:
#AQUÍ FORMO LA TABLA DE LO QUE TEORICAMENTE DEBIA PAGAR EL ESTUDIANTE 
#------------------------------------------------------------------------------------------------------------
Cedulas_filtroz = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
cedulas_para_zoho = ", ".join([f"'{cedulaz}'" for cedulaz in Cedulas_filtroz])
#------------------------------------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    ventas_zoho = pd.read_sql(f'''
SELECT DISTINCT
    doc_alum as Documento_Estudiante_zoho,
	PERIODO,
	FEC_PAGO_LIQ,
    VALOR_ORDEN_IC as VALOR_ORDEN,
    val_becdtos,
    tipo,
    ADICIONALES
FROM zoho.base_personas
WHERE
   ESTADO_PAGO = 'PAGO'
   AND EST_MATRICULADO = '1-Activo'
   AND DOCUMENTO IN ('FAMA','FECO')
   and PERIODO not like '%I%'
   AND doc_alum IN ({cedulas_para_zoho})
                               ''', engine2_1)
    ventas_zoho2 = ventas_zoho.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
ventas_zoho

Conexion a la base de datos realizada


Conexión cerrada


,Documento_Estudiante_zoho,PERIODO,FEC_PAGO_LIQ,VALOR_ORDEN,val_becdtos,tipo,ADICIONALES
0,1007495024,24V04,30/07/2024,3466000,-2467600,Pregrado,22000
1,1007495024,26V05,28/09/2026,3854800,0,Pregrado,None
2,1007495024,26V05,29/09/2026,4037200,-3710160,Pregrado,24000


In [11]:
ventas_zoho['valor_ingles'] = '0'
mask = ventas_zoho['tipo'] == 'Pregrado'
ventas_zoho.loc[mask, 'valor_ingles'] = '200000'

In [12]:
ventas_zoho = ventas_zoho.drop(columns=['tipo'])

In [13]:
#Se filtro el dataframe solo con las matriculas que corresponden a los mismo periodos que en etiquetas
ventas_zoho['llave_per_zoho']=ventas_zoho['Documento_Estudiante_zoho'].astype(str)+ventas_zoho['PERIODO'].astype(str)
ventas_zoho = ventas_zoho[ventas_zoho['llave_per_zoho'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
# ventas_zoho

In [14]:
ventas_zoho = ventas_zoho.drop('llave_per_zoho', axis=1)
#Reemplazamos vacios con 0 para no tener errores al momento de operar
columnas_valor = [
    'VALOR_ORDEN',
    'val_becdtos',
    'valor_ingles',
    'ADICIONALES'
]

for columna in columnas_valor:
    ventas_zoho[columna] = pd.to_numeric(
        ventas_zoho[columna],
        errors='coerce'
    ).fillna(0)

ventas_zoho['VALOR_A_PAGAR'] = (
    ventas_zoho['VALOR_ORDEN']
    + ventas_zoho['val_becdtos']
    + ventas_zoho['valor_ingles']
    + ventas_zoho['ADICIONALES']
)
ventas_zoho_t=ventas_zoho[['Documento_Estudiante_zoho','VALOR_ORDEN','val_becdtos','valor_ingles','ADICIONALES','VALOR_A_PAGAR']]
#Quitamos duplicados por cedula y tomamos el primer valor
df_resumen_zoho = ventas_zoho_t.drop_duplicates(subset='Documento_Estudiante_zoho', keep='first')

Aca ya la tabla de ventas no tiene matriculas duplicados por ende no traera multiples coincidencias

In [15]:
#Se concatena Etiquetas_mes_actual_t TODO LO DE ZOHO y lo de Recibos Caja para tener los valores pagados finales

Etiquetas_mes_actual_t=Etiquetas_mes_acutual[['Doc_Estudiante','PERIODO','Documento_del_Asesor','Periodo_Pago','TIPO','EstadoFinal']]
df_resultado1 = pd.merge(Etiquetas_mes_actual_t, df_resumen_zoho, left_on='Doc_Estudiante', right_on='Documento_Estudiante_zoho', how='left')
df_resultado2 = pd.merge(df_resultado1, df_resumen_recibos, left_on='Doc_Estudiante', right_on='DOC_ALUM', how='left')
df_resultado2.rename(columns={'VALOR_INGRESO_DIRECTO':'VALOR'},inplace=True)
df_resultado2.drop(columns=['Documento_Estudiante_zoho','DOC_ALUM'], inplace=True)

#---------Se divide la tabla por tipo para calcular las comisiones----------------------------------------
df_resultado2_preg=df_resultado2[df_resultado2['TIPO']=='Pregrado'] #PREGRADOS
df_resultado2_post=df_resultado2[df_resultado2['TIPO']=='Postgrado'] #POSTGRADOS
df_resultado2_Ec=df_resultado2[df_resultado2['TIPO']=='Educacion continuada'] #EDUC_CONTINUADA

In [16]:
df_resultado2.columns

Index(['Doc_Estudiante', 'PERIODO', 'Documento_del_Asesor', 'Periodo_Pago',
       'TIPO', 'EstadoFinal', 'VALOR_ORDEN', 'val_becdtos', 'valor_ingles',
       'ADICIONALES', 'VALOR_A_PAGAR', 'VALOR'],
      dtype='object')

ACA SON CALCULOS EXCLUSIVOS PARA EL DATAFRAME DE SOLO PREGRADOS

In [17]:
#-----------CALCULOS EXCLUSIVOS PREGRADO --df_resultado2_preg---------------------------------------
df_resultado2_preg['Pago_Total'] = np.where(df_resultado2_preg['VALOR'] == df_resultado2_preg['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
df_resultado2_preg['Modalidad'] = df_resultado2_preg['PERIODO'].apply(lambda x: 'VIRTUAL' if 'V' in x else 'PRESENCIAL')
almacen_etiquetas_preg_aprov=df_resultado2_preg.copy()
almacen_etiquetas_preg_aprov['Periodo_Pago_1']=PERIODO_ALMACEN

In [18]:
# SE CARGA LA INFORMACIÓN PARA LOS PREGRADOS QUE SE APRUEBAN A LIQUIDAR

from sqlalchemy import create_engine
from urllib.parse import quote_plus

host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'

# Driver ODBC
driver = quote_plus('ODBC Driver 17 for SQL Server')

# Cadena de conexión
url_2_12 = (
    f"mssql+pyodbc://@{host_2_12}:{port_2_12}/{database_2_12}"
    f"?trusted_connection=yes&driver={driver}"
)

# Crear conexión
engine_2_12 = create_engine(
    url_2_12,
    fast_executemany=True,
    pool_pre_ping=True
)

table_name_2_12 = 'RELIQ_PREGR_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'

try:

    print("🔄 Iniciando carga...")

    almacen_etiquetas_preg_aprov.to_sql(
        name=table_name_2_12,
        schema=schema_name_2_12,
        con=engine_2_12,
        if_exists='append',
        index=False,
        chunksize=10000
    )

    print("✅ Información cargada correctamente")

except Exception as e:

    print(f"❌ Error durante la carga:")
    print(e)

finally:

    engine_2_12.dispose()
    print("🔌 Conexión cerrada")

🔄 Iniciando carga...
✅ Información cargada correctamente
🔌 Conexión cerrada


In [19]:
#-----------CALCULOS EXCLUSIVOS Posgrados
df_resultado2_post['Pago_Total'] = np.where(df_resultado2_post['VALOR'] == df_resultado2_post['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
# df_resultado2_preg.to_excel('CuantoPago_Etiq_SoloPreg.xlsx')
almacen_etiquetas_post_aprov=df_resultado2_post.copy()
almacen_etiquetas_post_aprov['Periodo_Pago_1']=PERIODO_ALMACEN

In [20]:
#SE CARGA LA INFORMACION PARA LOS POSGRADOS QUE SE APRUEBAN A LIQUIDAR
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'RELIQ_POSG_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_post_aprov.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [21]:
#SE TOMA EL DATAFRAME DE PREGRADOS Y SE CUENTA POR CANTIDAD DE PAGOS PARCIALES, TOTALES, PRESENCIALES Y VIRTUALES

conteo_documentos = df_resultado2_preg['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos.columns = ['Documento_del_Asesor', 'Frecuencia']
def contar_condiciones(df, documento, pago_total, modalidad):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total) & (df['Modalidad'] == modalidad)])
conteo_documentos['P_parcial-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'PRESENCIAL'))
conteo_documentos['P_Total-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'PRESENCIAL'))
conteo_documentos['P_parcial-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'VIRTUAL'))
conteo_documentos['P_Total-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'VIRTUAL'))

In [22]:
#SE TOMA EL DATAFRAME DE POSGRADOS Y SE CUENTA POR CANTIDAD DE PAGOS PARCIALES, TOTALES, PRESENCIALES Y VIRTUALES

conteo_documentos_post = df_resultado2_post['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_post.columns = ['Documento_del_Asesor', 'Ventas_post']
def contar_condiciones_post(df, documento, pago_total):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total)])
conteo_documentos_post['P_parcial_post'] = conteo_documentos_post['Documento_del_Asesor'].apply(lambda x: contar_condiciones_post(df_resultado2_post, x, 'P_parcial_post'))
conteo_documentos_post['P_Total_post'] = conteo_documentos_post['Documento_del_Asesor'].apply(lambda x: contar_condiciones_post(df_resultado2_post, x, 'P_Total_post'))

In [23]:
almacen_etiquetas_preg_desglo_reliq_x=conteo_documentos.copy()
almacen_etiquetas_preg_desglo_reliq_x['Periodo_Pago_1']=PERIODO_ALMACEN

In [24]:
#--------------------------------------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'RELIQ_SOL_PREG_X'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_preg_desglo_reliq_x.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [25]:
almacen_etiquetas_post_desglo_reliq_x=conteo_documentos_post.copy()
almacen_etiquetas_post_desglo_reliq_x['Periodo_Pago_1']=PERIODO_ALMACEN

In [26]:
#--------------------------------------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'RELIQ_SOL_POST_X'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_post_desglo_reliq_x.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [27]:
#SE TOMAN LAS CEDULAS DE ASESORES QUE TIENEN ETIQUETAS DE RELIQUIDACION
Cedulas_Ase_LiqAnt = Etiquetas_mes_acutual['Documento_del_Asesor'].unique()
Cedulas_Ase_LiqAnt = ", ".join([f"'{cedulaAs}'" for cedulaAs in Cedulas_Ase_LiqAnt])


In [28]:
#ACA SE TOMA LA TABLA FIJA DESGLOSADA DE LA LIQUIDACION DEL MES PASADO SOLO CON LOS ASESORES
#QUE TIENEN ETIQUETAS
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    Desglos_MesAnterior12 = pd.read_sql(f'''
        Select * from  Comisiones_Contact.FIJA_LIQ_ACTUAL_DESGLOS 
                        where Periodo_Pago ={MES_SQL_SP_LIQ_ACTUAL} 
                        and Documento_Combinado IN({Cedulas_Ase_LiqAnt})
                               ''', engine2_1)
    Desglos_MesAnterior2 = Desglos_MesAnterior12.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
#-----------------------------------------------------
#EN LA MISMA CONSULTA DE DESGLOCE YO YA ESTOY FILTRANDO SOLO LAS CEDULAS DEL ASESOR QUE TIENE
#ETIQUETA DE RELIQUIDACION
Desglos_MesAnterior1=Desglos_MesAnterior12.copy()
Desglos_MesAnterior1 = Desglos_MesAnterior1.sort_values(by='Documento_Combinado', ascending=True)
Desglos_MesAnterior1
#----------------------------------------------
#----------------------------------------------

Conexion a la base de datos realizada
Conexión cerrada


,Documento_Combinado,Frecuencia,P_parcial-PRESENCIAL,P_Total-PRESENCIAL,P_parcial-VIRTUAL,P_Total-VIRTUAL,Vent_Parc_PRES,Vent_Total_PRES,Vent_Parc_VIRT,Vent_Total_VIRT,...,Ventas_post,Valor_Vent_post,Ventas_Ec,Valor_Vent_Ec,TOTAL_COMISION_GENERAL,Periodo_Pago,P_parcial_posg,P_Total_posg,Vent_Parc_posg,Vent_Total_posg
0,1016001445,59,0,0,58,1,0,0,3770000,120000,...,0,0,0,0,3890000,202609,0,0,0,0
1,1064804396,39,1,0,33,5,100000,0,1485000,400000,...,0,0,0,0,1985000,202609,0,0,0,0


## CALCULOS PREGRADOS

In [29]:
#SUMAR LA CANTIDAD DE VENTAS CON LA TABLA FIJA DE LA LIQUIDACION DEL MES PASADO DESGLOSADA
Pregr_Aprob_MesAnte=Desglos_MesAnterior1[['Documento_Combinado','Frecuencia','P_parcial-PRESENCIAL','P_Total-PRESENCIAL','P_parcial-VIRTUAL','P_Total-VIRTUAL']]
Pregr_Aprob_MesAnte['Documento_Combinado']=Pregr_Aprob_MesAnte['Documento_Combinado'].astype(str)
#ELIMINO POSIBLES DUPLICADOS DEL DESGLOCE DEL MES ANTERIOR
Pregr_Aprob_MesAnte = Pregr_Aprob_MesAnte.drop_duplicates(subset='Documento_Combinado', keep='first')
#---------------------------------------------------------------------------------
#LOS ASESORES CON ETIQUETAS DE RELIQ SE SUMA EL DESGLOCE DEL MES ANTERIOR - CRUCE NO ES OUTER
df_merged = pd.merge(conteo_documentos, Pregr_Aprob_MesAnte, left_on='Documento_del_Asesor', right_on='Documento_Combinado', 
how='left', suffixes=('_conteo_documentos', '_Pregr_Aprob_MesAnte'))
# Ahora sumamos las columnas correspondientes, manejando posibles NaN
df_merged['Frecuencia'] = df_merged['Frecuencia_Pregr_Aprob_MesAnte'].fillna(0) + df_merged['Frecuencia_conteo_documentos'].fillna(0)
df_merged['P_parcial-PRESENCIAL'] = df_merged['P_parcial-PRESENCIAL_Pregr_Aprob_MesAnte'].fillna(0) + df_merged['P_parcial-PRESENCIAL_conteo_documentos'].fillna(0)
df_merged['P_Total-PRESENCIAL'] = df_merged['P_Total-PRESENCIAL_Pregr_Aprob_MesAnte'].fillna(0) + df_merged['P_Total-PRESENCIAL_conteo_documentos'].fillna(0)
df_merged['P_parcial-VIRTUAL'] = df_merged['P_parcial-VIRTUAL_Pregr_Aprob_MesAnte'].fillna(0) + df_merged['P_parcial-VIRTUAL_conteo_documentos'].fillna(0)
df_merged['P_Total-VIRTUAL'] = df_merged['P_Total-VIRTUAL_Pregr_Aprob_MesAnte'].fillna(0) + df_merged['P_Total-VIRTUAL_conteo_documentos'].fillna(0)
# Ahora tenemos un dataframe con las columnas sumadas y los documentos correspondientes
df_merged['Documento_del_Asesor'] = df_merged['Documento_del_Asesor'].combine_first(df_merged['Documento_Combinado'])
df_final = df_merged[['Documento_del_Asesor','Frecuencia', 'P_parcial-PRESENCIAL', 'P_Total-PRESENCIAL', 'P_parcial-VIRTUAL', 'P_Total-VIRTUAL']]
conteo_documentos=df_final.copy() 
#df_final.to_excel('Sumafinal_preg_reliq.xlsx')#----VALIDADO Y QUEDO BIEN
#--------------------------------------
#--------------------------------------

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22704\3603037055.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Pregr_Aprob_MesAnte['Documento_Combinado']=Pregr_Aprob_MesAnte['Documento_Combinado'].astype(str)


In [30]:
#CALCULOS DE COMISION PARA PREGRADOS
#---------------------------------------------------------------------------------------
##conteo_documentos['Vent_Parc_PRES'] = 0
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 49980
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 66640
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 99960
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 116620
##conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 133280
##conteo_documentos['Vent_Total_PRES'] = 0
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 88000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 110000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 132000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 154000
##conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 176000
##conteo_documentos['Vent_Parc_VIRT'] = 0
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 45000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 55000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 65000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 75000
##conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 85000
##conteo_documentos['Vent_Total_VIRT'] = 0
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 55200
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 69000
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 82800
##conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 96600
##conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 110400
##conteo_documentos['Total_Com_Preg']= conteo_documentos['Vent_Parc_PRES'] + conteo_documentos['Vent_Total_PRES'] + conteo_documentos['Vent_Parc_VIRT'] + conteo_documentos['Vent_Total_VIRT']
#conteo_documentos.to_excel('Tabla_conteos_preg_total.xlsx')


In [31]:

#ACA HAGO LOS CALCULOS PARA LLENAR LAS COLUMNAS CON LAS MULTIPLICACIONES
#---------------------------------------------------------------------------------------
conteo_documentos['Vent_Parc_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 100000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 110000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 120000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 130000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 140000
conteo_documentos['Vent_Total_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 150000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 165000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 180000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 195000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 210000
conteo_documentos['Vent_Parc_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 45000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 55000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 65000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 75000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 85000
conteo_documentos['Vent_Total_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 80000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 100000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 120000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 140000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 150000
conteo_documentos['Total_Com_Preg']= conteo_documentos['Vent_Parc_PRES'] + conteo_documentos['Vent_Total_PRES'] + conteo_documentos['Vent_Parc_VIRT'] + conteo_documentos['Vent_Total_VIRT']
#conteo_documentos.to_excel('Tabla_conteos_preg_total.xlsx')

## CALCULOS PARA POSTGRADOS


In [32]:
#ACA DEBO SUMAR LA CANTIDAD DE VENTAS CON LA TABLA FIJA DE LA LIQUIDACION DEL MES PASADO DESGLOSADA
Post_Aprob_MesAnte=Desglos_MesAnterior1[['Documento_Combinado','Ventas_post','P_parcial_posg','P_Total_posg']]
Post_Aprob_MesAnte['Documento_Combinado']=Post_Aprob_MesAnte['Documento_Combinado'].astype(str)
#ELIMINO POSIBLES DUPLICADOS DEL DESGLOCE DEL MES ANTERIOR
Post_Aprob_MesAnte = Post_Aprob_MesAnte.drop_duplicates(subset='Documento_Combinado', keep='first')
#SIMPLEMENTE A LOS ASESORES CON ETIQUETAS DE RELIQ LES SUMO LO DEL DESGLOCE DEL MES ANTERIOR - CRUCE NO ES OUTER
df_merged_post = pd.merge(conteo_documentos_post, Post_Aprob_MesAnte, left_on='Documento_del_Asesor', right_on='Documento_Combinado', 
how='left', suffixes=('_conteo_documentos_post', '_Post_Aprob_MesAnte'))

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22704\338113511.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Post_Aprob_MesAnte['Documento_Combinado']=Post_Aprob_MesAnte['Documento_Combinado'].astype(str)


In [33]:
# Ahora sumamos las columnas correspondientes, manejando posibles NaN
df_merged_post['Ventas_post'] = df_merged_post['Ventas_post_Post_Aprob_MesAnte'].fillna(0) + df_merged_post['Ventas_post_conteo_documentos_post'].fillna(0)
df_merged_post['P_parcial_post'] = df_merged_post['P_parcial_posg'].fillna(0) + df_merged_post['P_parcial_post'].fillna(0)
df_merged_post['P_Total_post'] = df_merged_post['P_Total_posg'].fillna(0) + df_merged_post['P_Total_post'].fillna(0)
# Ahora tenemos un dataframe con las columnas sumadas y los documentos correspondientes
df_merged_post['Documento_del_Asesor'] = df_merged_post['Documento_del_Asesor'].combine_first(df_merged_post['Documento_Combinado'])
df_final_post = df_merged_post[['Documento_del_Asesor','Ventas_post', 'P_parcial_post', 'P_Total_post']]
conteo_documentos_post=df_final_post.copy() 
#df_final.to_excel('Sumafinal_preg_reliq.xlsx')#----VALIDADO Y QUEDO BIEN

In [34]:
conteo_documentos_post['Vent_Parc_posg'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Parc_posg'] = conteo_documentos_post['P_parcial_post'] * 50400
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Parc_posg'] = conteo_documentos_post['P_parcial_post'] * 64800
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Parc_posg'] = conteo_documentos_post['P_parcial_post'] * 79200
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Parc_posg'] = conteo_documentos_post['P_parcial_post'] * 93600
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Parc_posg'] = conteo_documentos_post['P_parcial_post'] * 108000
conteo_documentos_post['Vent_Total_posg'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Total_posg'] = conteo_documentos_post['P_Total_post'] * 112000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Total_posg'] = conteo_documentos_post['P_Total_post'] * 144000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Total_posg'] = conteo_documentos_post['P_Total_post'] * 176000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Total_posg'] = conteo_documentos_post['P_Total_post'] * 208000
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Total_posg'] = conteo_documentos_post['P_Total_post'] * 240000
conteo_documentos_post['Valor_Vent_post']= conteo_documentos_post['Vent_Parc_posg'] + conteo_documentos_post['Vent_Total_posg'] 

In [35]:
conteo_documentos_post

,Documento_del_Asesor,Ventas_post,P_parcial_post,P_Total_post,Vent_Parc_posg,Vent_Total_posg,Valor_Vent_post


## CALCULO EDUCACION CONTINUADA

In [36]:
conteo_documentos_Ec = df_resultado2_Ec['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_Ec.columns = ['Documento_del_Asesor_Ec', 'Ventas_Ec']

Ec_Aprob_MesAnte=Desglos_MesAnterior1[['Documento_Combinado','Ventas_Ec']]
Ec_Aprob_MesAnte['Documento_Combinado']=Ec_Aprob_MesAnte['Documento_Combinado'].astype(str)

#SIMPLEMENTE A LOS ASESORES CON ETIQUETAS DE RELIQ LES SUMO LO DEL DESGLOCE DEL MES ANTERIOR - CRUCE NO ES OUTER
df_merged_Ec = pd.merge(conteo_documentos_Ec, Ec_Aprob_MesAnte, left_on='Documento_del_Asesor_Ec', right_on='Documento_Combinado', 
how='left', suffixes=('_conteo_documentos', '_Pregr_Aprob_MesAnte'))
#sumamos
df_merged_Ec['Ventas_Ec'] = df_merged_Ec['Ventas_Ec_Pregr_Aprob_MesAnte'].fillna(0) + df_merged_Ec['Ventas_Ec_conteo_documentos'].fillna(0)
# Ahora tenemos un dataframe con las columnas sumadas
df_merged_Ec['Documento_del_Asesor_Ec'] = df_merged_Ec['Documento_del_Asesor_Ec'].combine_first(df_merged_Ec['Documento_Combinado'])
df_final_Ec = df_merged_Ec[['Documento_del_Asesor_Ec','Ventas_Ec']]

conteo_documentos_Ec=df_final_Ec.copy()
#-------------------------------------------------------
def calcular_valor_ventas_Ec(cantidad):
    if cantidad == 0:
        return 0
    elif cantidad >= 1:
        return cantidad * 10000
conteo_documentos_Ec['Valor_Vent_Ec'] = conteo_documentos_Ec['Ventas_Ec'].apply(calcular_valor_ventas_Ec)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22704\1903244935.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Ec_Aprob_MesAnte['Documento_Combinado']=Ec_Aprob_MesAnte['Documento_Combinado'].astype(str)


In [37]:
#ACA AÑADO AL DATAFRAME DE PREGRADOS POSTGRADOS - CRUCE SI DEBE SER OUTER (CC ASESOR PREG con CC ASESOR POST)
#------------------------------------------------------------------
resultado_m = pd.merge(conteo_documentos, conteo_documentos_post, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor', how='outer')
resultado_m['Documento_Combinado'] = resultado_m['Documento_del_Asesor'].combine_first(resultado_m['Documento_del_Asesor'])
resultado_m.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor'], inplace=True)
resultado_m=resultado_m.fillna(0)
columnas = resultado_m.columns.tolist()
columnas = ['Documento_Combinado'] + [col for col in columnas if col != 'Documento_Combinado']
resultado_m = resultado_m[columnas]
resultado_m.rename(columns={'Documento_Combinado': 'Documento_del_Asesor'}, inplace=True)
#resultado_m.to_excel('Merge_totalC_C.xlsx')
#------------------------------------------------------------------
#ACA AÑADO AL DATAFRAME DE UNIDO EDUCACION CONTINUADA - CRUCE SI DEBE SER OUTER (CC ASESOR PREG Y POST con CC ASESOR EC)
#------------------------------------------------------------------
resultado_f = pd.merge(resultado_m, conteo_documentos_Ec, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor_Ec', how='outer')
resultado_f['Documento_Combinado'] = resultado_f['Documento_del_Asesor'].combine_first(resultado_f['Documento_del_Asesor_Ec'])
resultado_f.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor_Ec'], inplace=True)
resultado_f=resultado_f.fillna(0)
columnas1 = resultado_f.columns.tolist()
columnas1 = ['Documento_Combinado'] + [col1 for col1 in columnas1 if col1 != 'Documento_Combinado']
resultado_f = resultado_f[columnas1]
#------------------------------------------------------------------
#ENCUENTRO EL TOTAL TOTAL
#------------------------------------------------------------------
resultado_f['TOTAL_COMISION_GENERAL']=resultado_f['Total_Com_Preg']+resultado_f['Valor_Vent_post']+resultado_f['Valor_Vent_Ec']
#resultado_f.to_excel('suma_final_antes_de_los_merge.xlsx')
#------------------------------------------------------------------
#------------------------------------------------------------------

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22704\4058052638.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  resultado_m=resultado_m.fillna(0)


In [38]:
resultado_f

,Documento_Combinado,Frecuencia,P_parcial-PRESENCIAL,P_Total-PRESENCIAL,P_parcial-VIRTUAL,P_Total-VIRTUAL,Vent_Parc_PRES,Vent_Total_PRES,Vent_Parc_VIRT,Vent_Total_VIRT,Total_Com_Preg,Ventas_post,P_parcial_post,P_Total_post,Vent_Parc_posg,Vent_Total_posg,Valor_Vent_post,Ventas_Ec,Valor_Vent_Ec,TOTAL_COMISION_GENERAL
0,1016001445,60,0,0,59,1,0,0,4425000,140000,4565000,0.0,0,0,0.0,0.0,0.0,0.0,0.0,4565000.0
1,1064804396,40,1,0,34,5,110000,0,1870000,500000,2480000,0.0,0,0,0.0,0.0,0.0,0.0,0.0,2480000.0


ACA EMPIEZO A ARMAR LA TABLA Comisiones_Contact.BasePagoReliquidaciones

In [39]:
#ACA DEBO TRAERME SOLO PARA LAS CEDULAS QUE HAY EN 'resultado_f' la liquidacion propia pasada
# y las Afectacioens pasadas
# ojo no es OUTER ninguna de esos merge, sino solo para las cedulas que estan aca para reliquidar

In [40]:
#ACA ME TRIGO LA BASE PAGO DEL MES DE LA RELIQUIDACION ES DECIR 202410 (MES ANTERIOR)
Cedulas_Comb = resultado_f['Documento_Combinado'].unique()
cedulas_consultaC = ", ".join([f"'{cedulac}'" for cedulac in Cedulas_Comb])
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    BasePago_Ant = pd.read_sql(f'''
        Select * from CUN_REPOSITORIO.COMISIONES_CONTACT.BasePago
                WHERE Periodo_Pago ={MES_SQL_SP_LIQ_ACTUAL} 
                and Documento_del_Asesor  IN ({cedulas_consultaC})
                               ''', engine2_1)
    BasePago_Ant2 = BasePago_Ant.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
BasePago_Ant


Conexion a la base de datos realizada
Conexión cerrada


,Documento_del_Asesor,Periodo_Pago,Pregrados,Postgrado,Educacion_continuada,Valor_Pregrado,Valor_Postgrado,Valor_Educacion_continuada,Total_Comision,Calidad_Resultado,...,Adherencia_Resultado,Conversion_Resultado,Por_Afectacion,Deducciones,Comisiones_Total_Deducciones,Nombres,email_Administrativo_he,Reliquidacion,Comision_Final,Productividad_Resultado
0,1016001445,202609.0,59,0,0,3890000,0,0,3890000,0.0,...,0.0,0.0,0.0,0,3890000,NORVEY IBARRA MARQUEZ,norvey_ibarra@cun.edu.co,0,3890000,0.0
1,1064804396,202609.0,39,0,0,1985000,0,0,1985000,0.0,...,0.0,0.0,0.0,0,1985000,CRISTIAN CAMILO SANCHEZ ARIAS,cristian_sanchez@cun.edu.co,575000,2560000,0.0


In [41]:
#ACA ME TRIGO las AFECTACIONES DEL MES DE LA RELIQUIDACION ES DECIR 202410 (MES ANTERIOR)
Cedulas_AFEC = resultado_f['Documento_Combinado'].unique()
cedulas_consulta_AF = ", ".join([f"'{cedulAfec_1}'" for cedulAfec_1 in Cedulas_AFEC])
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    Afectaciones_R = pd.read_sql(f'''
        Select * from CUN_REPOSITORIO.COMISIONES_CONTACT.Afectaciones_Contact 
                WHERE Periodo ={MES_SQL_SP_LIQ_ACTUAL} 
                and Documento_del_Asesor IN({cedulas_consulta_AF})
                               ''', engine2_1)
    Afectaciones_R2 = Afectaciones_R.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
Afectaciones_R

Conexion a la base de datos realizada


Conexión cerrada


,Documento_del_Asesor,Correo_Asesor,CALIDAD,FORMACION,VENTAS,ADHERENCIA,CONVERSION,PERIODO,PRODUCTIVIDAD


In [42]:
#--------------------------------------------------------------------------
#ACA AÑADO A LA TABLA FINAL, LA BASE PAGO
resultado_f = pd.merge(resultado_f, BasePago_Ant, left_on='Documento_Combinado', right_on='Documento_del_Asesor', how='left')
resultado_f = resultado_f.drop(columns=['Documento_del_Asesor'])
#---------------------ACA BORRO SOLO LAS COLUMNAS DE AFECTACIONES
resultado_f.drop(columns=['Calidad_Resultado', 'Formacion_Resultado','Ventas_Resultado','Adherencia_Resultado','Conversion_Resultado','Productividad_Resultado','Por_Afectacion'], inplace=True)
#-------------------------------------------------------------------------
#aca arreglo la tabla en valores
#-------------------------------------------------------------------------
Afectaciones_R.fillna(0, inplace=True)
Afectaciones_R.replace(['Cumple','CUMPLE','cumple'], 0,inplace=True)
Afectaciones_R.replace(['No Cumple','NO CUMPLE','No cumple','no cumple','no Cumple', 'NO CUMPLE '], peso_calificacion_ojo, inplace=True)
Afectaciones_R.drop(columns=['PERIODO'], inplace=True)
Afectaciones_R['Por_Afectacion'] = Afectaciones_R['CALIDAD']+Afectaciones_R['FORMACION ']+Afectaciones_R['VENTAS']+Afectaciones_R['ADHERENCIA']+Afectaciones_R['CONVERSION']+Afectaciones_R['PRODUCTIVIDAD']
#Afectaciones_R.to_excel('afectaciones_transformadas.xlsx')
#-------------------------------------------------------------------------
#ACA AÑADO A LA TABLA FINAL, LAS AFECTACIONES
#-------------------------------------------------------------------------
resultado_f = pd.merge(resultado_f, Afectaciones_R, left_on='Documento_Combinado', right_on='Documento_del_Asesor', how='left')
resultado_f['Periodo_Pago']=PERIODO_NORMAL_float
resultado_f.fillna(0, inplace=True)
resultado_f.drop(columns=['Documento_del_Asesor'], inplace=True)
#resultado_f.to_excel('con_base_pago_y_afectaciones.xlsx')
#------------------------
#------------------------

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22704\843707739.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  resultado_f.fillna(0, inplace=True)


In [43]:
resultado_f['Per_desglo']=PERIODO_ALMACEN
#------------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'RELIQ_DESGLO_BASE'
schema_name_2_12 = 'COMISIONES_CONTACT'
# Send the data to sql server
resultado_f.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()


## CALCULO DE LA RELIQUIDACION

In [44]:
resultado_f['Pregrados_Reliquidacion'] = resultado_f['Frecuencia'] - resultado_f['Pregrados']
resultado_f['Pregrados_Reliquidacion'] = resultado_f['Pregrados_Reliquidacion'].apply(lambda x: 0 if x < 0 else x)
resultado_f['Postgrado_Reliquidacion'] = resultado_f['Ventas_post'] - resultado_f['Postgrado']
resultado_f['Postgrado_Reliquidacion'] = resultado_f['Postgrado_Reliquidacion'].apply(lambda x: 0 if x < 0 else x)
resultado_f['Educacion_continuada_Reliquidacion'] = resultado_f['Ventas_Ec'] - resultado_f['Educacion_continuada']
resultado_f['Educacion_continuada_Reliquidacion'] = resultado_f['Educacion_continuada_Reliquidacion'].apply(lambda x: 0 if x < 0 else x)
#------------------------------------------------------------
resultado_f['Deducciones_Reliquidacion']=resultado_f['TOTAL_COMISION_GENERAL']*resultado_f['Por_Afectacion']
resultado_f['Total_Comision_Reliquidacion_Afectacion'] = resultado_f['TOTAL_COMISION_GENERAL'] - resultado_f['Deducciones_Reliquidacion']
resultado_f['Comision_Reliquidacion']=resultado_f['Total_Comision_Reliquidacion_Afectacion'] - resultado_f['Comision_Final']
resultado_f['Comision_Reliquidacion'] = resultado_f['Comision_Reliquidacion'].apply(lambda x: 0 if x < 0 else x)

In [45]:
#AQUI YA FORMO LA TABLA QUE NECESITA EL SP
#-----------------------------------------------------------------
Tabla_Conteos_SP=resultado_f.copy()

Tabla_Conteos_SP.rename(columns={'Documento_Combinado': 'Documento_del_Asesor',
                            'CALIDAD': 'Calidad_Resultado',
                            'FORMACION ': 'Formacion_Resultado',
                            'VENTAS': 'Ventas_Resultado',
                            'ADHERENCIA': 'Adherencia_Resultado',
                            'CONVERSION': 'Conversion_Resultado',
                            'PRODUCTIVIDAD':'Productividad_Resultado',
                            'Frecuencia':'Pregrados_Reliquidar',
                            'Ventas_post':'Postgrado_Reliquidar',
                            'Ventas_Ec':'EducacionContinuada_Reliquidar',
                            'Total_Com_Preg': 'Valor_Pregrado_reliquidacion',
                            'Valor_Vent_post': 'Valor_Postgrado_reliquidacion',
                            'Valor_Vent_Ec':'Valor_Educacion_continuada_reliquidacion',
                            'TOTAL_COMISION_GENERAL':'Total_Comision_Reliquidacion'

                            }, inplace=True)

In [46]:
Tabla_Conteos_SP=Tabla_Conteos_SP[['Documento_del_Asesor','Periodo_Pago',
                        'Pregrados', 'Postgrado','Educacion_continuada','Valor_Pregrado',
                        'Valor_Postgrado',	'Valor_Educacion_continuada',	'Total_Comision', 
                        'Calidad_Resultado',	'Formacion_Resultado',	'Ventas_Resultado',	
                        'Adherencia_Resultado',	'Conversion_Resultado',	'Por_Afectacion',	
                        'Deducciones',	'Comisiones_Total_Deducciones',	'Nombres',	
                        'email_Administrativo_he',	'Reliquidacion',	'Comision_Final',	
                        'Productividad_Resultado','Pregrados_Reliquidacion','Postgrado_Reliquidacion',
                        'Educacion_continuada_Reliquidacion','Pregrados_Reliquidar','Postgrado_Reliquidar',	
                        'EducacionContinuada_Reliquidar',	'Valor_Pregrado_reliquidacion',	
                        'Valor_Postgrado_reliquidacion','Valor_Educacion_continuada_reliquidacion',	
                        'Total_Comision_Reliquidacion','Deducciones_Reliquidacion',
                        'Total_Comision_Reliquidacion_Afectacion','Comision_Reliquidacion']]

Aqui ya no es replace sino APPEND

In [47]:
Tabla_Conteos_SP['Por_Afectacion'] = Tabla_Conteos_SP['Por_Afectacion'].replace(0.21000000000000002, 0.21)


In [48]:
Tabla_Conteos_SP['Deducciones_Reliquidacion']=Tabla_Conteos_SP['Deducciones_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Periodo_Pago']=Tabla_Conteos_SP['Periodo_Pago'].astype(float)

In [49]:
Tabla_Conteos_SP['Total_Comision_Reliquidacion_Afectacion']=Tabla_Conteos_SP['Total_Comision_Reliquidacion_Afectacion'].astype(int)
Tabla_Conteos_SP['Comision_Reliquidacion']=Tabla_Conteos_SP['Comision_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Total_Comision_Reliquidacion']=Tabla_Conteos_SP['Total_Comision_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Total_Comision_Reliquidacion']=Tabla_Conteos_SP['Total_Comision_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Valor_Pregrado_reliquidacion']=Tabla_Conteos_SP['Valor_Pregrado_reliquidacion'].astype(int)
Tabla_Conteos_SP['Valor_Postgrado_reliquidacion']=Tabla_Conteos_SP['Valor_Postgrado_reliquidacion'].astype(int)
Tabla_Conteos_SP['Documento_del_Asesor']=Tabla_Conteos_SP['Documento_del_Asesor'].astype(str)
Tabla_Conteos_SP['Pregrados_Reliquidacion']=Tabla_Conteos_SP['Pregrados_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Postgrado_Reliquidar']=Tabla_Conteos_SP['Postgrado_Reliquidar'].astype(int)
Tabla_Conteos_SP['EducacionContinuada_Reliquidar']=Tabla_Conteos_SP['EducacionContinuada_Reliquidar'].astype(int)
Tabla_Conteos_SP['Valor_Educacion_continuada_reliquidacion']=Tabla_Conteos_SP['Valor_Educacion_continuada_reliquidacion'].astype(int)
Tabla_Conteos_SP['Postgrado_Reliquidacion']=Tabla_Conteos_SP['Postgrado_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Educacion_continuada_Reliquidacion']=Tabla_Conteos_SP['Educacion_continuada_Reliquidacion'].astype(int)
Tabla_Conteos_SP['Pregrados_Reliquidar']=Tabla_Conteos_SP['Pregrados_Reliquidar'].astype(int)

In [50]:

host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'BasePagoReliquidaciones'
schema_name_2_12 = 'COMISIONES_CONTACT'
#Send the data to sql server
Tabla_Conteos_SP.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()



In [51]:
from sqlalchemy import create_engine, text

# Parámetros de conexión
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC+Driver+17 for SQL Server'.format(
    host=host_2_12, port=port_2_12, db=database_2_12
)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)


with engine_2_12.connect() as connection:
    try:
        sql_query = text(f'''
            DELETE FROM Comisiones_Contact.BasePago WHERE Periodo_Pago = {MES_ACTUAL};

            INSERT INTO Comisiones_Contact.BasePago (
                Documento_del_Asesor,
                Periodo_Pago,
                Pregrados,
                Postgrado,
                Educacion_continuada,
                Valor_Pregrado,
                Valor_Postgrado,
                Valor_Educacion_continuada,
                Total_Comision,
                Calidad_Resultado,
                Formacion_Resultado,
                Ventas_Resultado,
                Adherencia_Resultado,
                Conversion_Resultado,
                Por_Afectacion,
                Deducciones,
                Comisiones_Total_Deducciones,
                Nombres,
                email_Administrativo_he,
                Reliquidacion,
                Comision_Final,
                Productividad_Resultado
            )
            SELECT
                t3.Documento_del_Asesor,
                t3.Periodo_Pago,
                t3.Pregrados,
                t3.Postgrado,
                t3.Educacion_continuada,
                t3.Valor_Pregrado,
                t3.Valor_Postgrado,
                t3.Valor_Educacion_continuada,
                t3.Total_Comision,
                t3.Calidad_Resultado,
                t3.Formacion_Resultado,
                t3.Ventas_Resultado,
                t3.Adherencia_Resultado,
                t3.Conversion_Resultado,
                t3.Por_Afectacion,
                t3.Deducciones,
                t3.Total_Comision - t3.Deducciones AS Comisiones_Total_Deducciones,
                CONCAT(t4.nombre_btpr, ' ', t4.segundo_nombre_btpr, ' ', t4.primer_apellido_btpr, ' ', t4.segundo_apellido_btpr) AS Nombres,
                t4.email_Administrativo_he,
                ISNULL(t5.Comision_Reliquidacion, 0) AS Reliquidacion,
                ISNULL((t3.Total_Comision - t3.Deducciones) + ISNULL(t5.Comision_Reliquidacion, 0), 0) AS Comision_Final,
                t3.Productividad_Resultado
            FROM (
                SELECT
                    t1.*,
                    ISNULL(t2.Calidad_Resultado, 0) AS Calidad_Resultado,
                    ISNULL(t2.Formacion_Resultado, 0) AS Formacion_Resultado,
                    ISNULL(t2.Ventas_Resultado, 0) AS Ventas_Resultado,
                    ISNULL(t2.Adherencia_Resultado, 0) AS Adherencia_Resultado,
                    ISNULL(t2.Conversion_Resultado, 0) AS Conversion_Resultado,
                    ISNULL(t2.Por_Afectacion, 0) AS Por_Afectacion,
                    ISNULL(CONVERT(INT, t1.Total_Comision * t2.Por_Afectacion), 0) AS Deducciones,
                    ISNULL(t2.Productividad_Resultado, 0) AS Productividad_Resultado
                FROM CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS t1
                LEFT JOIN CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES t2
                    ON t1.Documento_del_Asesor = t2.Documento_del_Asesor AND t1.Periodo_Pago = t2.Periodo
            ) t3
            LEFT JOIN dbo.HUB_persona t4
                ON t3.Documento_del_Asesor = t4.identificacion_btpr
            LEFT JOIN (
                SELECT * FROM Comisiones_Contact.BasePagoReliquidaciones
                WHERE Periodo_Pago = {PERIODO_ALMACEN}
            ) t5
                ON t3.Documento_del_Asesor = t5.Documento_del_Asesor
            ORDER BY ISNULL((t3.Total_Comision - t3.Deducciones) + ISNULL(t5.Comision_Reliquidacion, 0), 0) DESC;
        ''')

        connection.execute(sql_query)
        connection.commit()
        print("Consulta realizada con éxito")

    except Exception as e:
        print(f"Error al ejecutar la consulta: {e}")

    finally:
        connection.close()


Consulta realizada con éxito


In [52]:
##with engine.connect() as connection:
##    try:
##
##        sql_query = text(f'''
##        Delete from Comisiones_Contact.BasePago where Periodo_Pago = {MES_ACTUAL}
##
##            Insert Into Comisiones_Contact.BasePago
##            (Documento_del_Asesor
##            ,Periodo_Pago
##            ,Pregrados
##            ,Postgrado
##            ,Educacion_continuada
##            ,Valor_Pregrado
##            ,Valor_Postgrado
##            ,Valor_Educacion_continuada
##            ,Total_Comision
##            ,Calidad_Resultado
##            ,Formacion_Resultado
##            ,Ventas_Resultado
##            ,Adherencia_Resultado
##            ,Conversion_Resultado
##            ,Por_Afectacion
##            ,Deducciones
##            ,Comisiones_Total_Deducciones
##            ,Nombres
##            ,email_Administrativo_he
##            ,Reliquidacion
##            ,Comision_Final
##            ,Productividad_Resultado)
##
##            Select T3.Documento_del_Asesor,
##            T3.Periodo_Pago,
##            T3.Pregrados,
##            T3.Postgrado,
##            T3.Educacion_continuada,
##            T3.Valor_Pregrado,
##            T3.Valor_Postgrado,
##            T3.Valor_Educacion_continuada,
##            T3.Total_Comision,
##            T3.Calidad_Resultado,
##            T3.Formacion_Resultado,
##            T3.Ventas_Resultado,
##            T3.Adherencia_Resultado,
##            T3.Conversion_Resultado,
##            T3.Por_Afectacion,
##            T3.Deducciones,
##            t3.Total_Comision-t3.Deducciones as Comisiones_Total_Deducciones,
##            concat(t4.nombre_btpr,' ',t4.segundo_nombre_btpr,' ',t4.primer_apellido_btpr,' ',t4.segundo_apellido_btpr) Nombres,
##            t4.email_Administrativo_he,
##            isnull(t5.Comision_Reliquidacion,0) as Reliquidacion,
##            isnull((t3.Total_Comision-t3.Deducciones)+isnull(t5.Comision_Reliquidacion,0),0) as Comision_Final,
##            T3.Productividad_Resultado
##                
##            from (
##            select t1.*,
##                isnull(t2.Calidad_Resultado,0) Calidad_Resultado,
##                isnull(t2.Formacion_Resultado,0) Formacion_Resultado,
##                isnull(t2.Ventas_Resultado,0) Ventas_Resultado,
##                isnull(t2.Adherencia_Resultado,0) Adherencia_Resultado,
##                isnull(t2.Conversion_Resultado,0) Conversion_Resultado,
##                isnull(t2.Por_Afectacion,0) Por_Afectacion,
##                isnull(convert(int,T1.Total_Comision*T2.Por_Afectacion),0) Deducciones,
##                isnull(t2.Productividad_Resultado,0) Productividad_Resultado
##            from CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS t1
##            left join CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES t2 on t1.Documento_del_Asesor = t2.Documento_del_Asesor and t1.Periodo_Pago = t2.Periodo)t3
##            left join dbo.HUB_persona t4 on t3.Documento_del_Asesor = t4.identificacion_btpr
##            left join (select * from Comisiones_Contact.BasePagoReliquidaciones
##                                where Periodo_Pago = {MES_ACTUAL}
##                                )t5 on t3.Documento_del_Asesor = t5.Documento_del_Asesor
##            order by (t3.Total_Comision-t3.Deducciones)+t5.Comision_Reliquidacion desc
##          
##        ''')
##        connection.execute(sql_query)
##        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
##        print("Consulta realizada con exito")
##    except Exception as e:
##        print(f"Error al eliminar registros: {e}")
##    finally:
##        connection.close()

In [53]:
mes_actual

10